<hr style="border:2px solid #0281c9"> </hr>

<img align="left" alt="ESO Logo" src="http://archive.eso.org/i/esologo.png">  

<div align="center">
  <h1 style="color: #0281c9; font-weight: bold;">ESO Science Archive</h1> 
  <h2 style="color: #0281c9; font-weight: bold;">Jupyter Notebooks</h2>
</div>

<hr style="border:2px solid #0281c9"> </hr>

# **Exploring GRAVITY Products with `astroquery.eso`**

This notebook demonstrates how to use `astroquery.eso` to discover **GRAVITY** data products around a given sky position, find the assosiated raw data, download both the data and the raw data, retrieve the associated ancillary preview files, download those previews, and finally display the previews side-by-side for rapid visual inspection.

---

### **Background**

This notebook explores data from the ESO Phase 3 **GRAVITY** collection:

- 📄 Official release description:  
  https://www.eso.org/rm/api/v1/public/releaseDescriptions/242

- 🌐 Science Portal collection page:  
  https://archive.eso.org/scienceportal/home?data_collection=GRAVITY

GRAVITY is a second-generation VLTI instrument operating with four telescopes (UTs or ATs) and delivering optical/near-infrared interferometric observations. The Phase 3 stream release includes:

- **Uncalibrated dispersed visibilities** for both SCIENCE and CALIBRATOR targets  
- **Calibrated science visibilities** (released in a second phase)  
- **Display plots and preview images** for rapid quality assessment  

In the first phase of the release, the primary products are **uncalibrated visibilities**, where detector and instrumental effects have been removed, but atmospheric and transfer-function calibration has not yet been applied. These products are delivered as **OIFITS2** files and are fully ESO Phase 3 compliant.

The release is a growing stream beginning with data from October 2016 onward and processed with pipeline version ≥1.9.

Full technical details of the release are described in the official release document. 

---

### **Preview Products**

Each GRAVITY Phase 3 dataset includes ancillary **preview plots** designed to allow quick inspection of:

- Squared visibilities (VIS2) for all six baselines  
- Visibility phases (VISPHI)  
- Closure phases (T3PHI)  
- Flux levels for the Fringe Tracker (FT) and Science Channel (SC)  
- (u,v) coverage  
- Quality-control flag summaries  

These preview plots are automatically generated by the GRAVITY pipeline and provide a compact visual summary of data quality.

For example, see the dataset page (used in this example):

- https://archive.eso.org/dataset/ADP.2025-12-17T14:16:43.439 

The plots displayed on that page are the same type of preview products that this notebook retrieves and displays programmatically.

---

### **What This Notebook Does**

This notebook demonstrates how to:

- Resolve a target name to sky coordinates using `SkyCoord.from_name`
- Query the ESO Science Archive for nearby **GRAVITY** Phase 3 data products
- Retrieve associated raw data products linked to selected `dp_id` values
- Download raw and reduced data products from the ESO Data Portal
- Retrieve associated ancillary preview products (e.g. `ANCILLARY.PREVIEW`) linked to selected `dp_id` values
- Download preview files from the ESO Data Portal
- Display preview images side-by-side for rapid comparison

This allows quick visual inspection of interferometric data quality without manually browsing individual dataset pages.

---

### **Compatibility Note**

This notebook uses a small helper module (`helpers_gravity.py`) to ensure compatibility with environments where `Eso.query_ancillary()` is not yet available in the installed `astroquery` release. Make sure you download this, and place it in the same directory as this notebook, to ensure it runs correctly.

---

### **Required `astroquery` Version**

This notebook requires the most recent development version of `astroquery` from the `main` branch. You can, for example, 
install this in a new conda environment with the following commands:

```bash
    conda create -n astroquery python=3.14 -y
    conda activate astroquery
    pip install git+https://github.com/astropy/astroquery.git@main
    pip install numpy, matplotlib, astropy, requests, tqdm, Pillow
```
<hr style="border:2px solid #0281c9"> </hr>

# **1. Setup and Imports**

Import the core libraries used in this workflow and print package versions to document the runtime environment.


In [ ]:
# Use postponed evaluation of type hints (PEP 563).
# This means annotations are stored as strings and not evaluated at runtime,
# making forward references and imports cleaner. Enabled by default in Python ≥3.11.
from __future__ import annotations

import sys

# --- Third-party scientific libraries ---
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image

# --- Astropy (astronomy-specific) ---
from astropy import units as u
from astropy.coordinates import SkyCoord

# --- Astroquery (ESO) ---
from astroquery.eso import Eso

# --- Helpers ---
from helpers.gravity import prepare_eso

# Print key package and python versions
print("Environment Information:\n")
print(f"{'Python':<12} {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
for pkg in ["numpy", "matplotlib", "astropy", "requests", "tqdm", "astroquery"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg:<12} {mod.__version__}")
    except ImportError:
        print(f"{pkg:<12} not installed")

# **2. Resolve Target Coordinates**

Resolve the target name to sky coordinates (RA, Dec) and define the cone-search radius. These values are used as the spatial constraint for archive queries.

In this case we use ``"FS CMA"`` as the target name (note also below this have other names such as ``HD 45677``). 

In [ ]:
target = "FS CMA"
coords = SkyCoord.from_name(target)
ra = coords.ra.deg 
dec = coords.dec.deg 
radius = (20*u.arcsec).to("deg").value

# **3. Query GRAVITY Products**

Initialize the ESO client and run a GRAVITY Phase 3 survey query around the target position. This returns the base product table used in later steps.

Note here the ``prepare_eso`` call, this is from the ``helpers_gravity.py`` module and is used as the current version of `astroquery` does not have the `query_ancillary` method is not yet available.

In [ ]:
# Note: here we don't do the standard eso = Eso() to allow for the helper function to modify the class before instantiation.
# This is due to query_ancillary not being implemented in the main branch at the time of writing...
eso = prepare_eso(Eso)
eso.ROW_LIMIT = 2 # limited to only 2 results
# eso.ROW_LIMIT = None # set to None to retrieve all results 

table = eso.query_surveys("GRAVITY", 
                          cone_ra=ra, 
                          cone_dec=dec, 
                          cone_radius=radius, 
                          )


**Optional - Apply query filters**

Just as an interesting example of what astroquery can do, we can make filters on the data. 

You can see in the help what are the columns available to filter on.


In [ ]:
eso.query_surveys("GRAVITY", help=True)

For example, we can define an optional proposal/programme ID filter for narrowing the selected products. 

In [ ]:
proposal_id = "like '113.268%'" 
column_filters = {'proposal_id': proposal_id}

# note that this still uses the eso.ROW_LIMIT = 2 set above
table_wfilters = eso.query_surveys("GRAVITY", 
                          cone_ra=ra, 
                          cone_dec=dec, 
                          cone_radius=radius,
                          column_filters=column_filters,
                          )

# **4. Download Data Products**

Download the selected products to a local directory.

In [ ]:
eso.retrieve_data(table["dp_id"], destination="./data")

# **5. Inspect Headers**

Retrieve headers for the selected products to inspect metadata and provenance links to raw data (for example `PROV*` columns). 

This could also be used to inspect any pipeline flags or quality-control indicators that may be present in the metadata... 


In [ ]:
table_headers = eso.get_headers(table["dp_id"])
colnames = table_headers.colnames
print("Available header columns:")
colnames[:10] # only print the first 10

# **6. Download Raw Files**

Identify provenance columns (``PROV*`` names) in the header table and download the linked files to the local `./data` directory.

In [ ]:
# TODO: FITS extension to be removed
# test
colnames_prov = [col for col in colnames if col.startswith("PROV")]
print(f"Provenance columns: {colnames_prov}")

for colname in colnames_prov:
    eso.retrieve_data(table_headers[colname], destination="./data")

# **7. Query and Download Ancillary Previews**

Query ancillary products with category `ANCILLARY.PREVIEW`, then download the preview files. These previews are the quicklook images visualized at the end.


In [ ]:
eso.ROW_LIMIT = None
table_ancillary = eso.query_ancillary(table["dp_id"],
                                     column_filters={"eso_category": "ANCILLARY.PREVIEW"})

# Assign output filenames to the ancillary table for later use in visualization
table_ancillary["filenames"] = eso.retrieve_data(table_ancillary["archive_id"], destination="./data/")

# **8. Quick Metadata Summary**

Print unique values of key metadata columns to sanity-check the result set. This helps confirm that targets, instruments, and programme IDs match expectations.

This is also interesting to note that the ``target_name`` differs from the ``FS CMA`` used in the query, but the coordinates match. This is a good reminder that target names are not standardized in the archive and can differ between datasets, so coordinates are often more reliable for confirming target identity.

In [ ]:
print_unique = ["proposal_id", "target_name", "instrument_name"]

for col in print_unique:
    print(f"\nUnique values for column '{col}':")
    print(np.unique(table[col]))

# **9. Display Preview Images**

Open downloaded preview files and plot them as side-by-side image pairs per product. This gives a fast visual overview of each GRAVITY dataset you just downloaded.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for row in table:
    product_id = row["dp_id"]
    target_name = row["target_name"]

    # select the two ancillary rows belonging to this product
    mask = table_ancillary["product_id"] == product_id
    ancillary_rows = table_ancillary[mask]

    if len(ancillary_rows) != 2:
        print(f"Skipping {product_id}: expected 2 ancillary files, got {len(ancillary_rows)}")
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    fig.suptitle(
        f"GRAVITY Data Previews – {product_id} – {target_name}",
        fontsize=16,
        fontweight="bold",
        y=0.9,
    )

    img1 = Image.open(ancillary_rows["filenames"][0])
    img2 = Image.open(ancillary_rows["filenames"][1])

    ax1.imshow(img1)
    ax2.imshow(img2)

    ax1.axis("off")
    ax2.axis("off")

    fig.tight_layout(w_pad=0)
    fig.subplots_adjust(wspace=-0.1)

## 10. Possible Additions

The material presented above provides a solid foundation for exploring GRAVITY Phase 3 data using `astroquery.eso`.

Users are encouraged to extend this notebook by appending additional analyses, diagnostic tools, or visualisations as appropriate. Possible extensions could include further inspection of OIFITS extensions, quality-control parameter summaries, or custom plotting.

Additional contributions and enhancements are welcome!

In [ ]:
print("\nAll done! You can now explore the downloaded data and ancillary files in the './data/' directory.")